Note: I ran two separate files for batch sizes 16 and 32 respectively due to limitation of GPU. I ran the complete code in two Kaggle notebooks. I have copied the code and results to Colab. I request you to kindly consider the lack of GPU resources available to us.

For Batch size=16:

In [ ]:
# Import required libraries
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, random_split
import time
import pandas as pd
from sklearn import svm
from sklearn.metrics import accuracy_score
from tqdm.auto import tqdm


In [ ]:
# Transform (converts 3 channels and resizes)
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

def get_dataloaders(dataset_name, batch_size, PinMem):
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.Grayscale(num_output_channels=3),
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
    ])

    # Load the full combined pool to get an accurate 70-10-20 split
    if dataset_name == "MNIST":
        full_ds = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
        test_ds = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)
    else:
        full_ds = torchvision.datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
        test_ds = torchvision.datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)

    # Combine them to treat as one pool for the 70-10-20 split requirement
    complete_dataset = torch.utils.data.ConcatDataset([full_ds, test_ds])
    total_size = len(complete_dataset)

    # Calculate sizes based on percentages (70%, 10%, 20%)
    train_size = int(0.7 * total_size)
    val_size = int(0.1 * total_size)
    test_size = total_size - train_size - val_size

    # Perform the split
    train_ds, val_ds, test_ds = random_split(complete_dataset, [train_size, val_size, test_size])

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, pin_memory=PinMem)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, pin_memory=PinMem)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, pin_memory=PinMem)

    return train_loader, val_loader, test_loader

In [ ]:
def run_resnet_experiment(model_name, dataset_name, bs, opt_type, lr, epochs, pm, exp_num):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    train_loader, _, test_loader = get_dataloaders(dataset_name, bs, pm)

    # pretrained=False requirement
    if model_name == "ResNet-18":
        model = torchvision.models.resnet18(weights=None)
    else:
        model = torchvision.models.resnet50(weights=None)

    model.fc = nn.Linear(model.fc.in_features, 10)
    model.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=lr) if opt_type == "SGD" else optim.Adam(model.parameters(), lr=lr)

    # Requirement: USE_AMP=True
    scaler = torch.amp.GradScaler('cuda')

    start_time = time.time()

    # Inner progress bar for epochs
    for epoch in range(epochs):
        model.train()
        train_loop = tqdm(train_loader, desc=f"Exp {exp_num} - Epoch {epoch+1}/{epochs}", leave=False)
        for images, labels in train_loop:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()

            with torch.amp.autocast('cuda'):
                outputs = model(images)
                loss = criterion(outputs, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

    # Evaluation
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    duration_ms = (time.time() - start_time) * 1000
    return accuracy, duration_ms

In [ ]:
# Setup Grid
datasets = ["MNIST", "FashionMNIST"]
models = ["ResNet-18", "ResNet-50"]
optimizers = ["SGD", "Adam"]
pin_mem=[True, False]
lrs = [0.001, 0.0001]
epochs_list = [3, 5]

resnet_results = []
c = 0

# Master Progress Bar
master_bar = tqdm(total=64, desc="Total Assignment Progress")

bs = 16
for ds in datasets:
    for m in models:
        for ep in epochs_list:
            for pm in pin_mem:
                for lr in lrs:
                    for opt in optimizers:
                        c += 1
                        print(f"Exp no.:{c}, Dataset:{ds}, Model:{m}, Batch size:{bs}, Optimizer:{opt}, Learning Rate:{lr}, Pin Memory:{pm}, No. of Epochs:{ep}")
                        acc, t_ms = run_resnet_experiment(m, ds, bs, opt, lr, ep, pm, c)
                        print(f"Accuracy:{acc}")
                        resnet_results.append({
                            "Dataset": ds, "Model": m, "Batch": bs, "Opt": opt,
                            "LR": lr, "Epochs": ep, "Pin memory": pm, "Accuracy": acc, 'Training time': t_ms
                        })

                        master_bar.update(1)
                        master_bar.set_postfix({"Last_Acc": f"{acc:.2f}%"})
master_bar.close()

# Save results
df = pd.DataFrame(resnet_results)
df.to_csv("q1a_16_results.csv", index=False)
print("All experiments complete. Results saved to q1a_16_results.csv.")

Total Assignment Progress:   0%|          | 0/64 [00:00<?, ?it/s]

Exp no.:1, Dataset:MNIST, Model:ResNet-18, Batch size:16, Optimizer:SGD, Learning Rate:0.001, Pin Memory:True, No. of Epochs:3



  0%|          | 0.00/9.91M [00:00<?, ?B/s]
  1%|          | 98.3k/9.91M [00:00<00:11, 822kB/s]
  4%|▍         | 393k/9.91M [00:00<00:05, 1.77MB/s]
 16%|█▌        | 1.61M/9.91M [00:00<00:01, 5.51MB/s]
100%|██████████| 9.91M/9.91M [00:00<00:00, 18.1MB/s]

100%|██████████| 28.9k/28.9k [00:00<00:00, 489kB/s]

  0%|          | 0.00/1.65M [00:00<?, ?B/s]
  6%|▌         | 98.3k/1.65M [00:00<00:01, 822kB/s]
 24%|██▍       | 393k/1.65M [00:00<00:00, 1.77MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.51MB/s]

100%|██████████| 4.54k/4.54k [00:00<00:00, 5.22MB/s]


Exp 1 - Epoch 1/3:   0%|          | 0/3063 [00:00<?, ?it/s]

Exp 1 - Epoch 2/3:   0%|          | 0/3063 [00:00<?, ?it/s]

...

Exp 50 - Epoch 1/3:   0%|          | 0/3063 [00:00<?, ?it/s]

Exp 50 - Epoch 2/3:   0%|          | 0/3063 [00:00<?, ?it/s]

Exp 50 - Epoch 3/3:   0%|          | 0/3063 [00:00<?, ?it/s]

Accuracy:90.1
Exp no.:51, Dataset:FashionMNIST, Model:ResNet-50, Batch size:16, Optimizer:SGD, Learning Rate:0.0001, Pin Memory:True, No. of Epochs:3


Exp 51 - Epoch 1/3:   0%|          | 0/3063 [00:00<?, ?it/s]

For Batch size=32:

In [ ]:
# Import required libraries
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, random_split
import time
import pandas as pd
from sklearn import svm
from sklearn.metrics import accuracy_score
from tqdm.auto import tqdm

In [ ]:
# Transform (converts 3 channels and resizes)
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

def get_dataloaders(dataset_name, batch_size, PinMem):
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.Grayscale(num_output_channels=3),
        transforms.ToTensor(),
        transforms.Normalize((0.5,), (0.5,))
    ])

    # Load the full combined pool to get an accurate 70-10-20 split
    if dataset_name == "MNIST":
        full_ds = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)
        test_ds = torchvision.datasets.MNIST(root='./data', train=False, download=True, transform=transform)
    else:
        full_ds = torchvision.datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
        test_ds = torchvision.datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)

    # Combine them to treat as one pool for the 70-10-20 split requirement
    complete_dataset = torch.utils.data.ConcatDataset([full_ds, test_ds])
    total_size = len(complete_dataset)

    # Calculate sizes based on percentages (70%, 10%, 20%)
    train_size = int(0.7 * total_size)
    val_size = int(0.1 * total_size)
    test_size = total_size - train_size - val_size

    # Perform the split
    train_ds, val_ds, test_ds = random_split(complete_dataset, [train_size, val_size, test_size])

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, pin_memory=PinMem)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, pin_memory=PinMem)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, pin_memory=PinMem)

    return train_loader, val_loader, test_loader


In [ ]:
def run_resnet_experiment(model_name, dataset_name, bs, opt_type, lr, epochs, pm, exp_num):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    train_loader, _, test_loader = get_dataloaders(dataset_name, bs, pm)

    # pretrained=False requirement
    if model_name == "ResNet-18":
        model = torchvision.models.resnet18(weights=None)
    else:
        model = torchvision.models.resnet50(weights=None)

    model.fc = nn.Linear(model.fc.in_features, 10)
    model.to(device)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=lr) if opt_type == "SGD" else optim.Adam(model.parameters(), lr=lr)

    # Requirement: USE_AMP=True
    scaler = torch.amp.GradScaler('cuda')

    start_time = time.time()

    # Inner progress bar for epochs
    for epoch in range(epochs):
        model.train()
        train_loop = tqdm(train_loader, desc=f"Exp {exp_num} - Epoch {epoch+1}/{epochs}", leave=False)
        for images, labels in train_loop:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()

            with torch.amp.autocast('cuda'):
                outputs = model(images)
                loss = criterion(outputs, labels)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

    # Evaluation
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    duration_ms = (time.time() - start_time) * 1000
    return accuracy, duration_ms

In [ ]:
# Setup Grid
datasets = ["MNIST", "FashionMNIST"]
models = ["ResNet-18", "ResNet-50"]
optimizers = ["SGD", "Adam"]
pin_mem=[True, False]
lrs = [0.001, 0.0001]
epochs_list = [3, 5]

resnet_results = []
c = 0

# Master Progress Bar
master_bar = tqdm(total=64, desc="Total Assignment Progress")

bs = 32
for ds in datasets:
    for m in models:
        for ep in epochs_list:
            for pm in pin_mem:
                for lr in lrs:
                    for opt in optimizers:
                        c += 1
                        print(f"Exp no.:{c}, Dataset:{ds}, Model:{m}, Batch size:{bs}, Optimizer:{opt}, Learning Rate:{lr}, Pin Memory:{pm}, No. of Epochs:{ep}")
                        acc, t_ms = run_resnet_experiment(m, ds, bs, opt, lr, ep, pm, c)
                        print(f"Accuracy:{acc}")
                        resnet_results.append({
                            "Dataset": ds, "Model": m, "Batch": bs, "Opt": opt,
                            "LR": lr, "Epochs": ep, "Pin memory": pm, "Accuracy": acc, 'Training time': t_ms
                        })

                        master_bar.update(1)
                        master_bar.set_postfix({"Last_Acc": f"{acc:.2f}%"})
master_bar.close()

# Save results
df = pd.DataFrame(resnet_results)
df.to_csv("q1a_32_results.csv", index=False)
print("All experiments complete. Results saved to q1a_32_results.csv.")

Total Assignment Progress:   0%|          | 0/64 [00:00<?, ?it/s]

Exp no.:1, Dataset:MNIST, Model:ResNet-18, Batch size:32, Optimizer:SGD, Learning Rate:0.001, Pin Memory:True, No. of Epochs:3



  0%|          | 0.00/9.91M [00:00<?, ?B/s]
  0%|          | 32.8k/9.91M [00:00<00:54, 181kB/s]
  1%|          | 65.5k/9.91M [00:00<00:54, 179kB/s]
  2%|▏         | 164k/9.91M [00:00<00:28, 342kB/s] 
  3%|▎         | 328k/9.91M [00:00<00:17, 560kB/s]
  7%|▋         | 655k/9.91M [00:00<00:09, 1.00MB/s]
 13%|█▎        | 1.31M/9.91M [00:01<00:04, 1.88MB/s]
 26%|██▌       | 2.56M/9.91M [00:01<00:02, 3.48MB/s]
 52%|█████▏    | 5.11M/9.91M [00:01<00:00, 6.81MB/s]
100%|██████████| 9.91M/9.91M [00:01<00:00, 5.98MB/s]

  0%|          | 0.00/28.9k [00:00<?, ?B/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 158kB/s]

  0%|          | 0.00/1.65M [00:00<?, ?B/s]
  2%|▏         | 32.8k/1.65M [00:00<00:08, 180kB/s]
  6%|▌         | 98.3k/1.65M [00:00<00:05, 282kB/s]
 10%|▉         | 164k/1.65M [00:00<00:04, 318kB/s] 
 22%|██▏       | 360k/1.65M [00:00<00:02, 615kB/s]
 44%|████▎     | 721k/1.65M [00:00<00:00, 1.10MB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.49MB/s]

100%|██████████| 4.54k/4.54k 

Exp 1 - Epoch 1/3:   0%|          | 0/1532 [00:00<?, ?it/s]

Exp 1 - Epoch 2/3:   0%|          | 0/1532 [00:00<?, ?it/s]

...

Exp 54 - Epoch 1/3:   0%|          | 0/1532 [00:00<?, ?it/s]

Exp 54 - Epoch 2/3:   0%|          | 0/1532 [00:00<?, ?it/s]

Exp 54 - Epoch 3/3:   0%|          | 0/1532 [00:00<?, ?it/s]

Accuracy:88.87142857142857
Exp no.:55, Dataset:FashionMNIST, Model:ResNet-50, Batch size:32, Optimizer:SGD, Learning Rate:0.0001, Pin Memory:False, No. of Epochs:3


Exp 55 - Epoch 1/3:   0%|          | 0/1532 [00:00<?, ?it/s]